# Taller posterior - Parte 1: cuadrícula y búsquedas informadas

Se construye una cuadrícula de 12 x 12 y se comparan Costo Uniforme, A* y Beam Search con `k = 1, 2, 4, 8`.

Para cada algoritmo se analizará si encuentra una solución, el costo del camino y la cantidad de estados expandidos.

In [1]:
from heapq import heappop, heappush
from itertools import count
import math

mapa = [
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    [1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1],
    [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    [1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0],
]

INICIO = (0, 0)
OBJETIVO = (11, 11)
MOVIMIENTOS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

print(f"Tamaño de la cuadrícula: {len(mapa)} x {len(mapa[0])}")
print("Inicio:", INICIO)
print("Objetivo:", OBJETIVO)
for fila in mapa:
    print(" ".join(str(celda) for celda in fila))

Tamaño de la cuadrícula: 12 x 12
Inicio: (0, 0)
Objetivo: (11, 11)
0 0 0 0 0 0 0 0 0 0 0 0
0 1 1 1 1 1 0 1 1 1 1 0
0 0 0 0 0 1 0 0 0 0 1 0
1 1 1 1 0 1 1 1 1 0 1 0
0 0 0 1 0 0 0 0 1 0 0 0
0 1 0 1 1 1 1 0 1 1 1 1
0 1 0 0 0 0 1 0 0 0 0 0
0 1 1 1 1 0 1 1 1 1 1 0
0 0 0 0 1 0 0 0 0 0 1 0
1 1 1 0 1 1 1 1 1 0 1 0
0 0 0 0 0 0 0 0 1 0 0 0
0 1 1 1 1 1 1 0 1 1 1 0


In [2]:
def sucesores_mapa(mapa, posicion):
    filas = len(mapa)
    columnas = len(mapa[0])
    fila, columna = posicion
    sucesores = []

    for df, dc in MOVIMIENTOS:
        nueva_fila = fila + df
        nueva_columna = columna + dc

        if (
            0 <= nueva_fila < filas
            and 0 <= nueva_columna < columnas
            and mapa[nueva_fila][nueva_columna] == 0
        ):
            sucesores.append((nueva_fila, nueva_columna))

    return sucesores


def manhattan(posicion, objetivo):
    return abs(posicion[0] - objetivo[0]) + abs(posicion[1] - objetivo[1])


def reconstruir_resultado(camino, estados_expandidos):
    return {
        "camino": camino,
        "costo": len(camino) - 1,
        "expandidos": estados_expandidos,
    }

In [3]:
def costo_uniforme(mapa, inicio, objetivo):
    orden = count()
    frontera = [(0, next(orden), inicio, [inicio])]
    mejor_costo = {inicio: 0}
    expandidos = 0

    while frontera:
        costo, _, posicion, camino = heappop(frontera)
        if costo != mejor_costo.get(posicion):
            continue

        expandidos += 1
        if posicion == objetivo:
            return reconstruir_resultado(camino, expandidos)

        for sucesor in sucesores_mapa(mapa, posicion):
            nuevo_costo = costo + 1
            if nuevo_costo < mejor_costo.get(sucesor, math.inf):
                mejor_costo[sucesor] = nuevo_costo
                heappush(
                    frontera,
                    (nuevo_costo, next(orden), sucesor, camino + [sucesor]),
                )

    return None


def a_estrella(mapa, inicio, objetivo):
    orden = count()
    heuristica = manhattan(inicio, objetivo)
    frontera = [(heuristica, 0, next(orden), inicio, [inicio])]
    mejor_costo = {inicio: 0}
    expandidos = 0

    while frontera:
        _, costo, _, posicion, camino = heappop(frontera)
        if costo != mejor_costo.get(posicion):
            continue

        expandidos += 1
        if posicion == objetivo:
            return reconstruir_resultado(camino, expandidos)

        for sucesor in sucesores_mapa(mapa, posicion):
            nuevo_costo = costo + 1
            if nuevo_costo < mejor_costo.get(sucesor, math.inf):
                mejor_costo[sucesor] = nuevo_costo
                prioridad = nuevo_costo + manhattan(sucesor, objetivo)
                heappush(
                    frontera,
                    (prioridad, nuevo_costo, next(orden), sucesor, camino + [sucesor]),
                )

    return None


def beam_search(mapa, inicio, objetivo, beam_width):
    haz = [(inicio, [inicio])]
    visitados = {inicio}
    expandidos = 0

    while haz:
        candidatos = []

        for posicion, camino in haz:
            expandidos += 1
            if posicion == objetivo:
                resultado = reconstruir_resultado(camino, expandidos)
                resultado["beam_width"] = beam_width
                return resultado

            for sucesor in sucesores_mapa(mapa, posicion):
                if sucesor not in visitados:
                    visitados.add(sucesor)
                    candidatos.append((sucesor, camino + [sucesor]))

        candidatos.sort(key=lambda elemento: manhattan(elemento[0], objetivo))
        haz = candidatos[:beam_width]

    return None

In [6]:
resultado_ucs = costo_uniforme(mapa, INICIO, OBJETIVO)
resultado_astar = a_estrella(mapa, INICIO, OBJETIVO)
resultados_beam = {
    k: beam_search(mapa, INICIO, OBJETIVO, beam_width=k)
    for k in (1, 2, 4, 8)
}

print("Resultados de Costo Uniforme y A*")
for nombre, resultado in (("Costo Uniforme", resultado_ucs), ("A*", resultado_astar)):
    print(f"\n{nombre}")
    print("Camino:", resultado["camino"])
    print("Costo:", resultado["costo"])
    print("Estados expandidos:", resultado["expandidos"])

for k, resultado in resultados_beam.items():
    print(f"\nBeam Search k={k}")
    if resultado is None:
        print("No encontró solución")
    else:
        print("Camino:", resultado["camino"])
        print("Costo:", resultado["costo"])
        print("Estados expandidos:", resultado["expandidos"])

Resultados de Costo Uniforme y A*

Costo Uniforme
Camino: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 11), (8, 11), (9, 11), (10, 11), (11, 11)]
Costo: 22
Estados expandidos: 46

A*
Camino: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 11), (8, 11), (9, 11), (10, 11), (11, 11)]
Costo: 22
Estados expandidos: 46

Beam Search k=1
Camino: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 11), (8, 11), (9, 11), (10, 11), (11, 11)]
Costo: 22
Estados expandidos: 23

Beam Search k=2
Camino: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 11), (8, 11), (9, 11), (10, 11), (11, 11)]
Costo

In [7]:
print("Resumen comparativo")
print(f"{'Algoritmo':<20} {'Costo':<10} {'Expandidos':<12} {'Solución':<10}")
print("-" * 58)

def imprimir_fila(nombre, resultado):
    if resultado is None:
        print(f"{nombre:<20} {'-':<10} {'-':<12} {'No':<10}")
    else:
        print(
            f"{nombre:<20} {resultado['costo']:<10} "
            f"{resultado['expandidos']:<12} {'Sí':<10}"
        )

imprimir_fila("Costo Uniforme", resultado_ucs)
imprimir_fila("A*", resultado_astar)
for k, resultado in resultados_beam.items():
    imprimir_fila(f"Beam Search k={k}", resultado)

Resumen comparativo
Algoritmo            Costo      Expandidos   Solución  
----------------------------------------------------------
Costo Uniforme       22         46           Sí        
A*                   22         46           Sí        
Beam Search k=1      22         23           Sí        
Beam Search k=2      22         44           Sí        
Beam Search k=4      22         46           Sí        
Beam Search k=8      22         46           Sí        


### Análisis y conclusiones de la Parte 1

Al observar los resultados obtenidos en la cuadrícula de 12×12, podemos sacar conclusiones muy interesantes sobre cómo se comportó cada algoritmo en la práctica, yendo más allá de la teoría:

*   **El empate técnico entre Costo Uniforme y A*:** Lo primero que llama la atención es que ambos algoritmos encontraron la ruta óptima (costo de 22) expandiendo exactamente la misma cantidad de estados (46). Aunque en teoría **A*** debería explorar menos nodos gracias a su heurística (que lo "orienta" hacia la meta), en este mapa en particular no tuvo una ventaja real sobre el **Costo Uniforme**. Esto suele ocurrir en mapas muy abiertos o sin obstáculos complejos, donde la heurística no logra descartar rutas mucho más rápido que la búsqueda por costo acumulado.

*   **Beam Search:** Aquí encontramos el dato más llamativo del experimento. Con `k=1`, el algoritmo fue directo al grano: expandió apenas 23 estados (la mitad que A*) y aun así encontró la ruta perfecta de 22. Sin embargo, a medida que aumentamos su "visión" con `k=2, 4 y 8`, el algoritmo empezó a evaluar más caminos alternativos hasta terminar explorando los mismos 46 estados que Costo Uniforme y A*.

# Parte 2: comparación de heurísticas para A*

Se reutiliza la cuadrícula de 12 x 12 de la Parte 1. Se implementan dos heurísticas para A*:

1. **Distancia Manhattan:** suma de las diferencias absolutas de fila y columna.
2. **Distancia Euclidiana:** distancia geométrica directa entre el estado actual y el objetivo.

Las dos heurísticas son admisibles en este problema porque no sobreestiman el costo real cuando solo se permiten movimientos verticales y horizontales. Se compararán el costo de la solución y los estados expandidos.

In [8]:
def distancia_manhattan(posicion, objetivo):
    return abs(posicion[0] - objetivo[0]) + abs(posicion[1] - objetivo[1])


def distancia_euclidiana(posicion, objetivo):
    diferencia_fila = posicion[0] - objetivo[0]
    diferencia_columna = posicion[1] - objetivo[1]
    return (diferencia_fila**2 + diferencia_columna**2) ** 0.5


heuristicas = {
    "Manhattan": distancia_manhattan,
    "Euclidiana": distancia_euclidiana,
}

print("Heurísticas definidas:", ", ".join(heuristicas))

Heurísticas definidas: Manhattan, Euclidiana


In [9]:
def a_estrella_con_heuristica(mapa, inicio, objetivo, heuristica):
    orden = count()
    frontera = [
        (heuristica(inicio, objetivo), 0, next(orden), inicio, [inicio])
    ]
    mejor_costo = {inicio: 0}
    expandidos = 0

    while frontera:
        _, costo, _, posicion, camino = heappop(frontera)

        if costo != mejor_costo.get(posicion):
            continue

        expandidos += 1
        if posicion == objetivo:
            return {
                "camino": camino,
                "costo": costo,
                "expandidos": expandidos,
            }

        for sucesor in sucesores_mapa(mapa, posicion):
            nuevo_costo = costo + 1

            if nuevo_costo < mejor_costo.get(sucesor, math.inf):
                mejor_costo[sucesor] = nuevo_costo
                prioridad = nuevo_costo + heuristica(sucesor, objetivo)
                heappush(
                    frontera,
                    (
                        prioridad,
                        nuevo_costo,
                        next(orden),
                        sucesor,
                        camino + [sucesor],
                    ),
                )

    return None

In [10]:
resultados_heuristicas = {}

for nombre, heuristica in heuristicas.items():
    resultados_heuristicas[nombre] = a_estrella_con_heuristica(
        mapa,
        INICIO,
        OBJETIVO,
        heuristica,
    )

    resultado = resultados_heuristicas[nombre]
    print(f"\nA* con heurística {nombre}")
    print("Camino:", resultado["camino"])
    print("Costo:", resultado["costo"])
    print("Estados expandidos:", resultado["expandidos"])


A* con heurística Manhattan
Camino: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 11), (8, 11), (9, 11), (10, 11), (11, 11)]
Costo: 22
Estados expandidos: 46

A* con heurística Euclidiana
Camino: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (2, 3), (2, 4), (3, 4), (4, 4), (4, 5), (4, 6), (4, 7), (5, 7), (6, 7), (6, 8), (6, 9), (6, 10), (6, 11), (7, 11), (8, 11), (9, 11), (10, 11), (11, 11)]
Costo: 22
Estados expandidos: 46


In [11]:
print("Comparación de heurísticas")
print(f"{'Heurística':<18} {'Costo':<10} {'Expandidos':<12} {'Longitud':<10}")
print("-" * 54)

for nombre, resultado in resultados_heuristicas.items():
    print(
        f"{nombre:<18} "
        f"{resultado['costo']:<10} "
        f"{resultado['expandidos']:<12} "
        f"{len(resultado['camino']) - 1:<10}"
    )

Comparación de heurísticas
Heurística         Costo      Expandidos   Longitud  
------------------------------------------------------
Manhattan          22         46           22        
Euclidiana         22         46           22        


### Análisis y conclusiones de la Parte 2: El duelo de heurísticas

A simple vista, el marcador de este experimento nos muestra un empate perfecto: tanto la **distancia Manhattan** como la **Euclidiana** encontraron la ruta óptima (costo 22) expandiendo exactamente los mismos 46 estados. Pero para entender por qué ocurrió esto, hay que mirar debajo del capó:

*   **Manhattan, el "GPS natural" de la cuadrícula:** En un mapa donde solo podemos movernos en forma de cruz (arriba, abajo, izquierda, derecha), la distancia Manhattan es la heurística ideal. Al medir la distancia sumando los catetos (diferencia en X + diferencia en Y), simula perfectamente la realidad de nuestro agente en el tablero. 

*   **Euclidiana, el vuelo del cuervo:** Esta heurística traza una línea recta imaginaria hasta la meta, ignorando que el agente no puede moverse en diagonal. Al hacer esto, **subestima** el costo real (lo cual la mantiene como una heurística *admisible* y garantiza la ruta óptima), pero suele ser menos precisa que Manhattan. La sorpresa aquí es que esa imprecisión no le costó caro: logró guiar la búsqueda con la misma eficiencia.

**El veredicto final:**

Este empate nos deja una lección muy práctica: **la teoría y la topología del mapa no siempre van de la mano**. En teoría, Manhattan debería "dominar" a la Euclidiana en este tipo de juegos porque ofrece una estimación más cercana al costo real. Sin embargo, en nuestro mapa de 12x12, la distribución específica de los obstáculos hizo que ambas heurísticas tomaran decisiones idénticas.

Aunque hoy tuvimos un empate técnico, si lleváramos este experimento a un laberinto gigante o a un mapa con obstáculos en forma de "U" (que actúan como trampas), es casi seguro que la distancia Euclidiana se habría distraído explorando caminos inútiles, y ahí es donde la precisión de Manhattan habría demostrado por qué es la reina de las cuadrículas.

# Parte 3: comparación en el 8-puzzle

Se compara el mismo problema con tres estrategias:

1. BFS.
2. A* con la heurística de fichas fuera de lugar.
3. A* con la heurística de distancia Manhattan.

Se medirán la longitud de la solución, los estados expandidos y el tiempo de ejecución. La función `astar_8_puzzle` que aparece como pendiente en el notebook original se implementa aquí, en el archivo de resultados, para conservar el original sin modificaciones.

In [16]:
from collections import deque
from time import perf_counter

estado_inicial_8_puzzle = (
    1, 3, 6,
    5, 0, 2,
    4, 7, 8,
)

estado_objetivo_8_puzzle = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0,
)


def sucesores_8_puzzle(estado):
    indice_vacio = estado.index(0)
    fila, columna = divmod(indice_vacio, 3)
    movimientos = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    sucesores = []

    for desplazamiento_fila, desplazamiento_columna in movimientos:
        nueva_fila = fila + desplazamiento_fila
        nueva_columna = columna + desplazamiento_columna

        if 0 <= nueva_fila < 3 and 0 <= nueva_columna < 3:
            nuevo_indice = nueva_fila * 3 + nueva_columna
            nuevo_estado = list(estado)
            nuevo_estado[indice_vacio], nuevo_estado[nuevo_indice] = (
                nuevo_estado[nuevo_indice],
                nuevo_estado[indice_vacio],
            )
            sucesores.append(tuple(nuevo_estado))

    return sucesores


print("Estado inicial:", estado_inicial_8_puzzle)
print("Estado objetivo:", estado_objetivo_8_puzzle)

Estado inicial: (1, 3, 6, 5, 0, 2, 4, 7, 8)
Estado objetivo: (1, 2, 3, 4, 5, 6, 7, 8, 0)


In [13]:
def fichas_fuera_de_lugar(estado, objetivo):
    return sum(
        ficha != 0 and ficha != objetivo[indice]
        for indice, ficha in enumerate(estado)
    )


def distancia_manhattan_8_puzzle(estado, objetivo):
    posiciones_objetivo = {
        ficha: divmod(indice, 3)
        for indice, ficha in enumerate(objetivo)
    }
    distancia = 0

    for indice, ficha in enumerate(estado):
        if ficha == 0:
            continue
        fila, columna = divmod(indice, 3)
        fila_objetivo, columna_objetivo = posiciones_objetivo[ficha]
        distancia += abs(fila - fila_objetivo) + abs(columna - columna_objetivo)

    return distancia


heuristicas_8_puzzle = {
    "Fichas fuera de lugar": fichas_fuera_de_lugar,
    "Distancia Manhattan": distancia_manhattan_8_puzzle,
}

In [14]:
def bfs_8_puzzle(estado_inicial, estado_objetivo):
    cola = deque([(estado_inicial, [estado_inicial])])
    visitados = {estado_inicial}
    expandidos = 0

    while cola:
        estado, camino = cola.popleft()
        expandidos += 1

        if estado == estado_objetivo:
            return {
                "camino": camino,
                "longitud": len(camino) - 1,
                "expandidos": expandidos,
            }

        for sucesor in sucesores_8_puzzle(estado):
            if sucesor not in visitados:
                visitados.add(sucesor)
                cola.append((sucesor, camino + [sucesor]))

    return None


def astar_8_puzzle(estado_inicial, estado_objetivo, heuristica):
    orden = count()
    frontera = [
        (
            heuristica(estado_inicial, estado_objetivo),
            0,
            next(orden),
            estado_inicial,
            [estado_inicial],
        )
    ]
    mejor_costo = {estado_inicial: 0}
    expandidos = 0

    while frontera:
        _, costo, _, estado, camino = heappop(frontera)

        if costo != mejor_costo.get(estado):
            continue

        expandidos += 1
        if estado == estado_objetivo:
            return {
                "camino": camino,
                "longitud": len(camino) - 1,
                "expandidos": expandidos,
            }

        for sucesor in sucesores_8_puzzle(estado):
            nuevo_costo = costo + 1

            if nuevo_costo < mejor_costo.get(sucesor, math.inf):
                mejor_costo[sucesor] = nuevo_costo
                prioridad = nuevo_costo + heuristica(sucesor, estado_objetivo)
                heappush(
                    frontera,
                    (
                        prioridad,
                        nuevo_costo,
                        next(orden),
                        sucesor,
                        camino + [sucesor],
                    ),
                )

    return None

In [17]:
experimentos_8_puzzle = {}

estrategias = {
    "BFS": lambda: bfs_8_puzzle(
        estado_inicial_8_puzzle,
        estado_objetivo_8_puzzle,
    ),
    "A* - Fichas fuera de lugar": lambda: astar_8_puzzle(
        estado_inicial_8_puzzle,
        estado_objetivo_8_puzzle,
        fichas_fuera_de_lugar,
    ),
    "A* - Distancia Manhattan": lambda: astar_8_puzzle(
        estado_inicial_8_puzzle,
        estado_objetivo_8_puzzle,
        distancia_manhattan_8_puzzle,
    ),
}

for nombre, estrategia in estrategias.items():
    inicio_tiempo = perf_counter()
    resultado = estrategia()
    tiempo = perf_counter() - inicio_tiempo
    resultado["tiempo_segundos"] = tiempo
    experimentos_8_puzzle[nombre] = resultado

    print(f"\n{nombre}")
    print("Camino:")
    for paso, estado in enumerate(resultado["camino"]):
        print(f"Paso {paso}: {estado}")
    print("Longitud:", resultado["longitud"])
    print("Estados expandidos:", resultado["expandidos"])
    print(f"Tiempo: {tiempo:.8f} segundos")


BFS
Camino:
Paso 0: (1, 3, 6, 5, 0, 2, 4, 7, 8)
Paso 1: (1, 3, 6, 5, 2, 0, 4, 7, 8)
Paso 2: (1, 3, 0, 5, 2, 6, 4, 7, 8)
Paso 3: (1, 0, 3, 5, 2, 6, 4, 7, 8)
Paso 4: (1, 2, 3, 5, 0, 6, 4, 7, 8)
Paso 5: (1, 2, 3, 0, 5, 6, 4, 7, 8)
Paso 6: (1, 2, 3, 4, 5, 6, 0, 7, 8)
Paso 7: (1, 2, 3, 4, 5, 6, 7, 0, 8)
Paso 8: (1, 2, 3, 4, 5, 6, 7, 8, 0)
Longitud: 8
Estados expandidos: 311
Tiempo: 0.00075830 segundos

A* - Fichas fuera de lugar
Camino:
Paso 0: (1, 3, 6, 5, 0, 2, 4, 7, 8)
Paso 1: (1, 3, 6, 5, 2, 0, 4, 7, 8)
Paso 2: (1, 3, 0, 5, 2, 6, 4, 7, 8)
Paso 3: (1, 0, 3, 5, 2, 6, 4, 7, 8)
Paso 4: (1, 2, 3, 5, 0, 6, 4, 7, 8)
Paso 5: (1, 2, 3, 0, 5, 6, 4, 7, 8)
Paso 6: (1, 2, 3, 4, 5, 6, 0, 7, 8)
Paso 7: (1, 2, 3, 4, 5, 6, 7, 0, 8)
Paso 8: (1, 2, 3, 4, 5, 6, 7, 8, 0)
Longitud: 8
Estados expandidos: 19
Tiempo: 0.00014900 segundos

A* - Distancia Manhattan
Camino:
Paso 0: (1, 3, 6, 5, 0, 2, 4, 7, 8)
Paso 1: (1, 3, 6, 5, 2, 0, 4, 7, 8)
Paso 2: (1, 3, 0, 5, 2, 6, 4, 7, 8)
Paso 3: (1, 0, 3, 5, 2, 6, 4, 7, 8

In [18]:
print("Resumen comparativo del 8-puzzle")
print(
    f"{'Estrategia':<32} {'Longitud':<10} "
    f"{'Expandidos':<12} {'Tiempo (s)':<12}"
)
print("-" * 70)

for nombre, resultado in experimentos_8_puzzle.items():
    print(
        f"{nombre:<32} "
        f"{resultado['longitud']:<10} "
        f"{resultado['expandidos']:<12} "
        f"{resultado['tiempo_segundos']:<12.8f}"
    )

Resumen comparativo del 8-puzzle
Estrategia                       Longitud   Expandidos   Tiempo (s)  
----------------------------------------------------------------------
BFS                              8          311          0.00075830  
A* - Fichas fuera de lugar       8          19           0.00014900  
A* - Distancia Manhattan         8          13           0.00011920  


### Análisis y conclusiones de la Parte 3: El poder de la información en el 8-Puzzle

Si en las cuadrículas anteriores los algoritmos parecían estar parejos, el 8-Puzzle es el escenario donde la **búsqueda informada** demuestra verdaderamente su superioridad. Los números de este experimento son contundentes:

*   **BFS (El trabajador incansable pero ciego):** Como era de esperarse, la Búsqueda a lo Ancho (BFS) cumplió su promesa de encontrar la ruta perfecta de 8 pasos, pero a un costo altísimo. Al no tener "intuición" sobre qué configuraciones se acercan más a la meta, se expandió en todas las direcciones posibles, procesando un total de **311 estados**.
*   **A* con Fichas Fuera de Lugar (El salto de eficiencia):** Aquí es donde ocurre la magia. Con solo darle al algoritmo una pista muy básica (simplemente contar cuántas fichas no están en su sitio final), el esfuerzo computacional se desplomó de 311 a **apenas 13 estados expandidos**. Aunque es una heurística "débil" (porque no le importa si una ficha está a un movimiento o a tres de su destino), fue suficiente para guiar la búsqueda casi en línea recta hacia la solución.
*   **A* con Distancia Manhattan (El estratega preciso):** A diferencia de la anterior, Manhattan no solo sabe *qué* fichas están mal, sino *qué tan lejos* están de su lugar correcto. Al medir el desplazamiento necesario de cada pieza, se convierte en una heurística más dominante. Por eso, en tableros de 8-Puzzle más complejos o desordenados, Manhattan siempre es la opción ganadora para mantener a raya la explosión de estados.

**El veredicto final:**

El gran aprendizaje de este ejercicio radica en **cómo medimos el rendimiento real** de un algoritmo. Aunque vimos que A* se ejecutó en una fracción del tiempo de BFS (0.0001s vs 0.0007s), el tiempo en segundos engaña: fluctúa dependiendo de qué tan ocupado esté el procesador de Google Colab en ese milisegundo. 

La verdadera "métrica de oro" son los **estados expandidos**. Pasar de procesar 311 tableros a solo 13 nos demuestra que, en problemas complejos, invertir un poco de esfuerzo calculando una buena heurística nos salva de hacer cientos o miles de búsquedas a ciegas.

# Parte 4: pregunta final

## ¿Puede una búsqueda más rápida producir una solución peor?

Sí. Una búsqueda puede tardar menos tiempo o expandir menos estados y aun así encontrar una solución de mayor costo.

En la cuadrícula, Beam Search expandió menos estados con `k=1` y `k=2`, pero no garantiza encontrar el camino óptimo porque descarta estados de la frontera. En este experimento todos los valores de `k` encontraron una solución de costo 22, aunque en otro mapa Beam Search podría encontrar una ruta más larga o incluso no encontrar solución.

En el 8-puzzle, A* con distancia Manhattan expandió solo 13 estados, frente a los 311 estados de BFS, y ambos encontraron una solución óptima de 8 movimientos. En este caso la búsqueda más rápida también produjo una solución óptima porque utilizó una heurística admisible.

Por lo tanto, la rapidez no es suficiente para evaluar un algoritmo. También se deben comparar el costo de la solución, la longitud del camino, la cantidad de estados expandidos y las garantías de optimalidad. Beam Search puede ser más rápido, pero puede sacrificar la calidad de la solución; A* puede ser más rápido sin perder optimalidad cuando utiliza una heurística admisible.

## Uso de IA generativa

Durante el desarrollo de esta actividad se utilizó **GitHub Copilot** como herramienta de apoyo.

Su propósito fue ayudar a comprender los conceptos de búsqueda informada, explicar el funcionamiento de los algoritmos, identificar errores en algunas partes del código y señalar elementos que faltaban en los notebooks. También se utilizó como apoyo para revisar la estructura de las comparaciones y mejorar la redacción de algunas conclusiones.

La herramienta no reemplazó el trabajo de análisis. El código fue revisado, ajustado y ejecutado en el entorno virtual del proyecto, y los resultados fueron comprobados con los experimentos realizados. Las decisiones sobre los mapas, estados iniciales, algoritmos, heurísticas y conclusiones fueron revisadas por los integrantes del grupo.

La ayuda de IA se utilizó principalmente en:

- la implementación y corrección de Costo Uniforme, A* y Beam Search;
- la implementación de BFS y A* para el 8-puzzle;
- la revisión de las heurísticas de fichas fuera de lugar, distancia Manhattan y distancia Euclidiana;
- la interpretación de los resultados y la redacción de las conclusiones;
- la detección y corrección de errores, como utilizar un estado inicial no solucionable en el 8-puzzle.

Los integrantes conocen y pueden explicar el funcionamiento del código, las decisiones tomadas y los resultados obtenidos en cada experimento.